In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

# Standard library imports
import argparse
import importlib.util
import inspect
import json
import math
import os
import pickle
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import warnings
from functools import reduce

# Third-party imports
import librosa
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from matplotlib import cm
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
import umap
from pathlib import Path
import cv2
from IPython.display import Audio, display

from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget, BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Local imports
import commons
import lightning_wrapper
import models
import utils
import train
from cough_datasets import (
    CoughDatasets,
    CoughDatasetsCollate,
    CoughDatasetsProcessorCollate,
    CoughDetectionRatioBatchSampler,
    CoughDiseaseBinaryBatchSampler,
    PatientBatchSampler
)

torch.set_float32_matmul_precision("medium")
cmap = cm.get_cmap("viridis")

parser = train.parse_args()
args = parser.parse_args(["--init", "--model_name", "dev", "--pooling_model",
                          "PEFTWavLM_Try1", "--feature_dim", "1024", "--config_path", "configs/general.json", 
                          "--use_precomputed", "--precomputed_dir", "./precomputed_features/logmel"])

model_dir = os.path.join("./logs", args.model_name)
os.makedirs(model_dir, exist_ok=True)
port = None

config_path = args.config_path if args.init else os.path.join(model_dir, "config.json")
hps = train.load_config(config_path, model_dir, args)

df_train, _ = train.load_data(hps)
collate_fn = train.get_collate_fn(hps)
target_labels = df_train[hps.data.target_column]

if not args.use_precomputed:
    utils.compute_spectrogram_stats_from_dataset(
        df_train, 
        hps.data, 
        pickle_path=f"{hps.model_dir}/wav_stats.pickle"
    )

train_loader, val_loader = train.prepare_fold_data(
    df_train, df_train, hps, "", collate_fn,
    use_precomputed=args.use_precomputed,
    precomputed_dir=args.precomputed_dir
)


In [ ]:
df_train[df_train['path_file'] == "/run/media/fourier/Data1/Pras/DatabaseLLM/coda/wavs/solicited_data/1636629193874-recording-1.wav"]

In [ ]:
batch = next(iter(train_loader))
wavnames, audio1, audio2, attention_masks, dse_ids, [patient_ids, _, tabular_ids, indice] = batch
labels = np.argmax(dse_ids.cpu().numpy(), axis=-1)

In [ ]:
batch = next(iter(val_loader))
wavnames, audio1, audio2, attention_masks, dse_ids, [patient_ids, _, tabular_ids, indice] = batch
labels = np.argmax(dse_ids.cpu().numpy(), axis=-1)

In [ ]:
selected_index = random.randint(0, len(audio1) - 1)
audio_np = audio1[selected_index].cpu().numpy().reshape(-1)

display(Audio(audio_np, rate=16000))
plt.plot(audio_np)

In [ ]:
plt.imshow(audio1[selected_index].cpu().numpy())

In [ ]:
audio_aug, applied_effects = train_dataset.data_augmentator(torch.from_numpy(audio_np).unsqueeze(0), train_dataset.sampling_rate, return_effects=True)
audio_aug = audio_aug.squeeze(0).numpy()

print(applied_effects)
display(Audio(audio_aug, rate=train_dataset.sampling_rate))
plt.plot(audio_aug)

In [ ]:
selected_index = random.randint(0, len(audio1) - 1)
audio_np = audio1[selected_index].cpu().numpy()

plt.imshow(audio_np)

In [ ]:
hola = torch.load("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/precomputed_features/train_1__run_media_fourier_Data1_Pras_DatabaseLLM_coda_wavs_solicited_data_1645088760390-recording-1.pt")